# Data Cleaning & Preprocessing: Student Testimony Dataset

## Project Overview

This notebook implements a comprehensive data cleaning pipeline to transform raw student testimonies into a high-quality, normalized dataset suitable for text mining and clustering analysis.

### Data Pipeline Stages

1. **Loading & Parsing**: Read raw CSV with complex formatting and extract text columns
2. **Quality Metrics**: Analyze data loss and quality before/after cleaning  
3. **Deduplication**: Identify and remove duplicate testimonies to avoid bias
4. **Filtering**: Remove common stop words that don't contribute semantic value
5. **Output**: Generate clean dataset for downstream analysis

### Data Quality Goals

- **Reduce Noise**: Eliminate non-informative terms cluttering the analysis
- **Preserve Signal**: Retain all semantically meaningful content
- **Enable Clustering**: Prepare data for unsupervised learning algorithms
- **Quality Metrics**: Track data retention at each processing stage


In [17]:
import pandas as pd


In [18]:

input_csv_path = "Student_Dataset.csv"
output_csv_path = "Cleaned_Student_Dataset.csv"

df_raw = pd.read_csv(
    input_csv_path,
    delimiter=",",
    header=None,
    index_col=0,
    dtype=str,
    names=["ID", "Colors", "Testimony"]
)

initial_record_count = len(df_raw)
print(f"✓ Dataset loaded successfully")
print(f"  Input file: {input_csv_path}")
print(f"  Initial records: {initial_record_count}")


✓ Dataset loaded successfully
  Input file: Student_Dataset.csv
  Initial records: 1011


## Step 2: Extract and Normalize Testimony Texts

### Objective
Parse raw CSV rows with complex formatting, extract clean testimony text, and handle edge cases like quoted strings and malformed entries.

### Methodology
- Line-by-line parsing to handle CSV inconsistencies
- Strip trailing newlines and whitespace
- Detect and remove color hex codes (e.g., `0x000000,`)
- Handle quoted strings with escaped quotes
- Filter empty or invalid testimonies

### Tools & Libraries
- `pandas`: For data structure management
- Built-in file I/O: For line-by-line parsing and processing


In [19]:
def extract_testimony_text(raw_line: str) -> str | None:
    raw_line = raw_line.rstrip("\n")

    if not raw_line:
        return None
    
    try:
        _record_id, rest_of_line = raw_line.split(",", 1)
    except ValueError:
        return None
    
    rest_of_line = rest_of_line.lstrip()
    if rest_of_line.startswith("0x000000,"):
        rest_of_line = rest_of_line[len("0x000000,"):].lstrip()

    rest_of_line = rest_of_line.strip()
    
    if len(rest_of_line) >= 2 and rest_of_line[0] == '"' and rest_of_line[-1] == '"':
        rest_of_line = rest_of_line[1:-1].replace('""', '"')

    return rest_of_line or None


extracted_testimonies = []
with open(input_csv_path, "r", encoding="utf-8") as file:
    for line in file:
        testimony_text = extract_testimony_text(line)
        if testimony_text:
            extracted_testimonies.append(testimony_text)

records_lost_during_parsing = initial_record_count - len(extracted_testimonies)
print(f"\n✓ Parsing complete")
print(f"  Testimonies extracted: {len(extracted_testimonies)}")
print(f"  Records lost (invalid format): {records_lost_during_parsing}")



✓ Parsing complete
  Testimonies extracted: 1011
  Records lost (invalid format): 0


9## Step 3: Remove Duplicate Testimonies

### Objective
Identify and eliminate duplicate testimonies to prevent sampling bias and ensure each unique voice is represented equally in downstream analysis.

### Methodology
- Use dictionary-based deduplication to preserve first occurrence order
- Compare full testimony text for exact matches
- Track duplication rate as quality metric

### Tools & Libraries
- Python `dict.fromkeys()`: Efficient deduplication preserving insertion order (Python 3.7+)


In [20]:
def remove_duplicates(testimony_list: list[str]) -> list[str]:
    return list(dict.fromkeys(testimony_list))


deduplicated_testimonies = remove_duplicates(extracted_testimonies)

duplicate_count = len(extracted_testimonies) - len(deduplicated_testimonies)
if len(extracted_testimonies)> 0:
    duplicate_rate = (duplicate_count / len(extracted_testimonies)) * 100
else:
    duplicate_rate = 0.0

print(f"\n✓ Deduplication complete")
print(f"  Unique testimonies: {len(deduplicated_testimonies)}")
print(f"  Duplicates removed: {duplicate_count}")
print(f"  Duplication rate: {duplicate_rate:.1f}%")



✓ Deduplication complete
  Unique testimonies: 1002
  Duplicates removed: 9
  Duplication rate: 0.9%


## Step 4: Stop Word Removal & Text Normalization

### Objective
Remove common English words and articles that don't carry semantic meaning, reducing noise while preserving content.

### Methodology
- Define application-specific stop word list (common English words + domain-specific terms)
- Convert text to lowercase for consistency
- Remove stop words while preserving meaningful terms
- Retain punctuation and multi-word phrases

### Example Transformation
**Before**: "I have a persistent feeling of thirst, especially after meals. I also lose weight without trying."

**After**: "persistent feeling thirst, especially after meals. lose weight without trying."

### Tools & Libraries
- Python set operations: Efficient O(1) lookups for stop word filtering


In [21]:
custom_stop_words = {
    'the', 'a', 'an', 'i', 'it', 'its', 'this', 'that',
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has',
    'to', 'of', 'in', 'for', 'with', 'on', 'at', 'by', 'from', 'up', 'down',
    'and', 'or', 'but', 'if', 'then', 'else',
    'stand', 'new', 'like'
}

cleaned_testimonies = []

for testimony in deduplicated_testimonies:
    words = testimony.lower().split()
    
    meaningful_words = [word for word in words if word not in custom_stop_words]

    cleaned_testimony = " ".join(meaningful_words)
    cleaned_testimonies.append(cleaned_testimony)

print(f"\n✓ Stop word filtering complete")
print(f"  Cleaned testimonies: {len(cleaned_testimonies)}")
print(f"  Stop words defined: {len(custom_stop_words)}")



✓ Stop word filtering complete
  Cleaned testimonies: 1002
  Stop words defined: 37


## Step 5: Create Final Clean Dataset

### Objective
Consolidate cleaned testimonies into a structured DataFrame and persist to CSV for downstream analysis.

### Structure
- **Index**: Sequential record identifier  
- **Color**: Placeholder column (0x000000 for all records)
- **Sentences**: Cleaned, deduplicated, filtered testimony text

### Quality Summary
Generates summary statistics on data retention and cleaning effectiveness across all preprocessing stages.


In [22]:
df_cleaned = pd.DataFrame({
    "Color": ["0x000000"] * len(cleaned_testimonies),
    "Sentences": cleaned_testimonies
})

final_record_count = len(df_cleaned)
total_records_lost = initial_record_count - final_record_count
retention_rate = (final_record_count / initial_record_count) * 100

print(f"\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)
print(f"Initial records (raw):        {initial_record_count:6d}")
print(f"Records lost (invalid):       {records_lost_during_parsing:6d}")
print(f"Records lost (duplicated):    {duplicate_count:6d}")
print(f"Final records (clean):        {final_record_count:6d}")
print(f"Total records lost:           {total_records_lost:6d}")
print(f"Data retention rate:          {retention_rate:6.1f}%")
print("=" * 70)



DATA QUALITY SUMMARY
Initial records (raw):          1011
Records lost (invalid):            0
Records lost (duplicated):         9
Final records (clean):          1002
Total records lost:                9
Data retention rate:            99.1%


## Step 6: Export Clean Dataset to CSV

### Output Format
- **Path**: `Cleaned_Student_Dataset.csv`
- **Structure**: Index (row number) | Color | Sentences (cleaned testimony)
- **Encoding**: UTF-8 for Unicode support


In [23]:
df_cleaned.to_csv(output_csv_path, index=True, index_label="Index", encoding="utf-8")

print(f"\n✓ Export complete")
print(f"  Output file: {output_csv_path}")
print(f"  Records exported: {len(df_cleaned)}")
print(f"  File ready for clustering analysis")



✓ Export complete
  Output file: Cleaned_Student_Dataset.csv
  Records exported: 1002
  File ready for clustering analysis
